# RMSNorm demo

(Replaces the RMSNorm half of `scripts/embedding_norm_demo.py`. Self-contained: only external libraries.)

**RMSNorm** rescales each token's vector so its typical size (root-mean-square) is 1:

    y = x * rsqrt(mean(x²) + eps) * weight

It keeps training stable, because every layer sees inputs of a similar scale. Only the *length* of the
vector changes, not its direction. The learned `weight` (one number per dimension, starting at 1) lets the
model rescale individual dimensions afterwards.

We use random vectors in place of real embeddings, since only the scale matters here.

Select the kernel **smalllm (.venv)** before running.

In [1]:
import torch
from torch import Tensor, nn

torch.manual_seed(0)

HIDDEN_SIZE = 128
EPS = 1e-5


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        # Normalise in float32 for stability, then cast back before applying the weight.
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * xf.to(x.dtype)


def rms(x: Tensor) -> list[float]:
    """Root-mean-square (typical size) of each token vector, for a (1, T, hidden) tensor."""
    return [round(v, 4) for v in x.pow(2).mean(-1).sqrt()[0].tolist()]

## 1. Some token vectors
Four random vectors of size 128 with rms about 0.02 (what a freshly initialised embedding gives).

In [2]:
vectors = torch.randn(1, 4, HIDDEN_SIZE) * 0.02
print("shape:", tuple(vectors.shape), "(batch, tokens, hidden)")
print("rms:  ", rms(vectors))

shape: (1, 4, 128) (batch, tokens, hidden)
rms:   [0.0208, 0.0165, 0.02, 0.0234]


## 2. Tokens of very different scale
Multiply the four vectors by 1, 10, 100 and 0.01, then normalise. All come out near rms 1. The last token is
so small that `eps` (a tiny constant that prevents dividing by zero) dominates, so it stays small.

In [3]:
norm = RMSNorm(HIDDEN_SIZE, EPS)

scales = torch.tensor([1.0, 10.0, 100.0, 0.01]).view(1, 4, 1)
scaled = vectors * scales
normed = norm(scaled)
print(f"rms before: {rms(scaled)}")
print(f"rms after:  {rms(normed)}")

rms before: [0.0208, 0.1652, 1.999, 0.0002]
rms after:  [0.9886, 0.9998, 1.0, 0.0739]


## 3. Check against the formula by hand
The learned weight starts at 1.0, so at the start the module is just `x * rsqrt(mean(x²) + eps)`.

In [4]:
by_hand = scaled * torch.rsqrt(scaled.pow(2).mean(-1, keepdim=True) + EPS)
print(f"matches the formula by hand: {torch.allclose(by_hand, normed)}")
print(f"initial norm weights (first 4): {norm.weight[:4].tolist()}")

matches the formula by hand: True
initial norm weights (first 4): [1.0, 1.0, 1.0, 1.0]


## 4. Direction is unchanged
Only the length changes, so the cosine similarity before and after is 1.

In [5]:
cosine = torch.cosine_similarity(scaled[0, 0], normed[0, 0], dim=0).item()
print(f"cosine similarity before/after for token 0: {cosine:.4f}")

cosine similarity before/after for token 0: 1.0000


## 5. Step by step on one vector
The same operation split into individual arithmetic steps, on token 0's 128 numbers.

In [6]:
x = vectors[0, 0].detach()  # shape (hidden,)
h = HIDDEN_SIZE


def first6(t: Tensor) -> list[float]:
    return [round(v, 4) for v in t[:6].tolist()]


print(f"x, first 6: {first6(x)}")

x, first 6: [-0.0225, -0.023, -0.005, -0.0087, 0.017, 0.0138]


**Steps 1–2:** square every number, then take the mean.

In [7]:
squares = x.pow(2)
print(f"1. squared, first 6:       {[f'{v:.6f}' for v in squares[:6].tolist()]}")
print(f"   sum of all {h} squares:  {squares.sum().item():.6f}")

mean_square = squares.mean()
print(f"2. mean square (sum / {h}): {mean_square.item():.8f}")

1. squared, first 6:       ['0.000507', '0.000531', '0.000025', '0.000075', '0.000288', '0.000192']
   sum of all 128 squares:  0.055401
2. mean square (sum / 128): 0.00043282


**Steps 3–4:** add `eps`, take the reciprocal square root to get a scale factor, and multiply every number by it.

In [8]:
with_eps = mean_square + EPS
print(f"3. + eps:                  {with_eps.item():.8f}")
print(f"   rms = sqrt of that:     {with_eps.sqrt().item():.6f}")

scale = torch.rsqrt(with_eps)
print(f"4. scale = 1 / rms:        {scale.item():.4f}")
y = x * scale
print(f"   x * scale, first 6:     {first6(y)}")
print(f"   y / x, first 6:         {first6(y / x)}  (the same factor for every number)")

3. + eps:                  0.00044282
   rms = sqrt of that:     0.021043
4. scale = 1 / rms:        47.5210
   x * scale, first 6:     [-1.07, -1.0952, -0.2382, -0.4124, 0.8066, 0.6577]
   y / x, first 6:         [47.521, 47.521, 47.521, 47.521, 47.521, 47.521]  (the same factor for every number)


**Before vs after:** the rms becomes 1, and the Euclidean length becomes √hidden.

In [9]:
print(f"rms:              {squares.mean().sqrt().item():.4f} -> {y.pow(2).mean().sqrt().item():.4f}")
print(f"largest |value|:  {x.abs().max().item():.4f} -> {y.abs().max().item():.4f}")
print(f"euclidean length: {x.norm().item():.4f} -> {y.norm().item():.4f}  (sqrt({h}) = {h**0.5:.4f})")

module_out = RMSNorm(h, EPS)(x)
print(f"RMSNorm module gives the same numbers: {torch.allclose(module_out, y, atol=1e-6)}")

rms:              0.0208 -> 0.9886
largest |value|:  0.0682 -> 3.2414
euclidean length: 0.2354 -> 11.1852  (sqrt(128) = 11.3137)
RMSNorm module gives the same numbers: True


## 6. The learned weight
After normalising, each dimension is multiplied by its own weight. Training can raise or lower individual
dimensions; here we set a few by hand to see the effect.

In [10]:
norm2 = RMSNorm(h, EPS)
with torch.no_grad():
    norm2.weight[:3] = torch.tensor([2.0, 0.0, 0.5])
out = norm2(x)
print("normalised only: ", first6(y)[:3])
print("with weights 2, 0, 0.5:", first6(out)[:3])

normalised only:  [-1.07, -1.0952, -0.2382]
with weights 2, 0, 0.5: [-2.14, -0.0, -0.1191]
